In [2]:
from tqdm.auto import tqdm
# import sys
# sys.path.append('/app')
from ingest import load_faq_data

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

import os
load_dotenv()
# openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


## Q1. Embedding a query
### Embed the following query:

### How does approximate nearest neighbor search work?

### The embedder returns a vector of 384 numbers. What's the first value (v[0])?


In [4]:
from embed.download import download
download("Xenova/all-MiniLM-L6-v2")

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [5]:
import onnxruntime as ort
import numpy as np
# import sys
# sys.path.append('/app/embed')

from embed.embedder import Embedder

embed = Embedder()
# ort_sess = ort.InferenceSession('all-MiniLM-L6-v2')

# onnx_model = onnx.load("all-MiniLM-L6-v2")
# onnx.checker.check_model(onnx_model)

In [6]:
query_q1 = "How does approximate nearest neighbor search work?"

In [7]:
v1 = embed.encode(query_q1)
v1[0]


np.float64(-0.02058203437252893)

## Q2. Cosine similarity
### The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

### Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [22]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "02-vector-search/lessons/07-sqlitesearch-vector" in path,
)

documents = [file.parse() for file in reader.read()]

In [27]:
documents[0]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [24]:
texts = [doc["content"] for doc in documents]

In [20]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/1 [00:00<?, ?it/s]

In [33]:
query_q1 = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query_q1)

scores = X.dot(v_query)
idx = np.argmax(scores)

#documents[idx]
top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]
for i in top_indices:
    print(f"{scores[i]:.4f}  {documents[i]['filename']}")


0.3611  02-vector-search/lessons/07-sqlitesearch-vector.md


NameError: name 'vec_to_str' is not defined

In [19]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self, embedder,**kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query: str, num_results: int = 5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}
        results = self.index.search(
            query_vector, filter_dict=filter_dict,
            num_results=num_results)
        
        return results